# Mapper-LLM Inference Notebook

Load a trained checkpoint and run inference in two modes:
- **Manual** — paste any source text and get a prediction
- **File** — run on a CSV / JSONL / Parquet file and optionally save results

In [ ]:
import sys, os
# Make sure the project root is on the path when running from any working directory
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), "."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import pandas as pd
from IPython.display import display

from src.pipeline import load_pipeline, predict
from src.metrics import bleu4, rouge_l

print("Imports OK")

## Configuration

Edit the variables below before running the rest of the notebook.

In [ ]:
# ── Required ────────────────────────────────────────────────────────────────
CHECKPOINT = "runs/pooled_qwen_trainable_emb_ctx16/version_0/checkpoints/last.ckpt"
CONFIG     = "configs/qwen_mlp_config.yaml"

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# ── Generation ──────────────────────────────────────────────────────────────
MAX_NEW_TOKENS      = 128
BATCH_SIZE          = 8
REPETITION_PENALTY  = 1.3

print(f"Device: {DEVICE}")

## Load Pipeline

In [ ]:
module, emb_tok, llm_tok = load_pipeline(CHECKPOINT, CONFIG, DEVICE)
print("Pipeline loaded.")

# Shared kwargs passed to every predict() call
_gen = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    repetition_penalty=REPETITION_PENALTY,
)

---
## Manual Inference

Paste any source text below. Set `TASK = "qa"` and fill in `QUESTION` for QA.

In [ ]:
SOURCE_TEXT = """
Paste your source text here.
"""

TASK     = "narrative"   # "narrative" | "qa"
QUESTION = ""            # only used when TASK == "qa"

records = [{"source_text": SOURCE_TEXT.strip(), "task": TASK, "question": QUESTION}]
pred    = predict(module, emb_tok, llm_tok, records, **_gen)[0]

print("=== Prediction ===")
print(pred)

---
## File-Based Inference

Input file must have columns: `source_text`, `task` (and optionally `question`, `answer`).

Supported formats: CSV, JSONL, Parquet.

In [ ]:
INPUT_FILE  = "data/my_examples.csv"   # ← path to your file
OUTPUT_FILE = "predictions.csv"        # ← set to None to skip saving

# ── Load file ───────────────────────────────────────────────────────────────
def read_file(path: str) -> pd.DataFrame:
    if path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    elif path.endswith(".json"):
        return pd.read_json(path)
    elif path.endswith(".parquet"):
        return pd.read_parquet(path)
    return pd.read_csv(path)

df = read_file(INPUT_FILE)
print(f"Loaded {len(df)} rows. Columns: {list(df.columns)}")
display(df.head(3))

In [ ]:
# ── Run inference ────────────────────────────────────────────────────────────
if "question" not in df.columns:
    df["question"] = ""

records = df[["source_text", "task", "question"]].fillna("").to_dict("records")

print(f"Running inference on {len(records)} examples …")
preds = predict(module, emb_tok, llm_tok, records, **_gen)

df_out = df.copy()
df_out["prediction"] = preds
print("Done.")

In [ ]:
# ── Display results ──────────────────────────────────────────────────────────
display_cols = [c for c in ["source_text", "task", "question", "answer", "prediction"]
                if c in df_out.columns]

pd.set_option("display.max_colwidth", 120)
display(df_out[display_cols].head(20))

In [ ]:
# ── Save results ─────────────────────────────────────────────────────────────
if OUTPUT_FILE is not None:
    df_out.to_csv(OUTPUT_FILE, index=False)
    print(f"Saved {len(df_out)} rows → {OUTPUT_FILE}")
else:
    print("OUTPUT_FILE is None — skipping save.")

---
## Metrics

Only runs if the input file contains an `answer` column (ground-truth references).
Reports ROUGE-L and BLEU-4 per task.

In [ ]:
if "answer" not in df_out.columns:
    print("No 'answer' column found — skipping metrics.")
else:
    # Обрезаем и pred, и ref до первых MAX_EVAL_WORDS слов перед подсчётом.
    # Без этого BLEU для narrative занижен в ~10x из-за штрафа за краткость (BP):
    # модель генерирует ~128 токенов, а референс — полный текст (300-500 слов).
    MAX_EVAL_WORDS = MAX_NEW_TOKENS  # 128

    def trunc(text, n):
        return " ".join(str(text).split()[:n])

    rows = []
    for task_name in sorted(df_out["task"].dropna().unique()):
        sub      = df_out[df_out["task"] == task_name]
        preds_t  = [trunc(p, MAX_EVAL_WORDS) for p in sub["prediction"]]
        refs_t   = [trunc(r, MAX_EVAL_WORDS) for r in sub["answer"]]

        r = rouge_l(preds_t, refs_t)
        b = bleu4(preds_t, refs_t)
        rows.append({"task": task_name, "n": len(sub),
                     "ROUGE-L": round(r["rougeL"], 4),
                     "BLEU-4":  round(b["bleu"],   6)})

    display(pd.DataFrame(rows).set_index("task"))

---
## Per-Task Sample Table

Side-by-side view of source / prediction / answer for quick qualitative inspection.

In [ ]:
N_SAMPLES = 5   # number of examples to show per task

pd.set_option("display.max_colwidth", 200)
for task_name in sorted(df_out["task"].dropna().unique()):
    sub  = df_out[df_out["task"] == task_name]
    show = [c for c in ["source_text", "question", "prediction", "answer"]
            if c in sub.columns]
    print(f"\n=== {task_name} (showing {min(N_SAMPLES, len(sub))} / {len(sub)}) ===")
    display(sub[show].head(N_SAMPLES).reset_index(drop=True))